# TripMate on Google Colab

**NOTE :- This is a non-editable notebook. Please create a copy of this Colab notebook before making any changes.**

This notebook runs your TripMate voice agent. The shared web page only connects a browser to whichever LiveKit project you paste in.

 Each student adds their own API keys in the Colab Secrets panel after creating a copy of this notebook.

---


## Before you run anything

1. Create a free project at [LiveKit Cloud](https://cloud.livekit.io).
2. Visit this document for detailed steps to **register with LiveKit and generate the API keys and token server ID**.
 [LiveKit registration guide](https://docs.google.com/document/d/1CIsnK0gTn3meFRhPW3P2dsjwgp2U0-HLdc-uSNPxA2k/edit?usp=sharing)
3. While registering with livekit make sure to turn **Development token server** on (Settings → Keys).
4. Add Colab Secrets (**Notebook access** ON):

| Secret name | Required |
| --- | --- |
| `LIVEKIT_URL` | yes |
| `LIVEKIT_API_KEY` | yes |
| `LIVEKIT_API_SECRET` | yes |
 Do not paste the token server ID into secrets — that ID goes in the UI website.
---

## Then
Run all the cells in order. The `download-files` cell must finish before you start the agent. Leave the agent cell running until you see `registered worker`, then paste your token server ID on the [TripMate site](https://tripmate-ai-blond.vercel.app).


## Workshop — three VoiceAgents

### The Interactive Roadmap

> *   **VoiceAgent 1:** We build a simple voice chatbot. It speaks, but it happily lies about the prices because it has no tools.
> *   **VoiceAgent 2:** We try to fix it using instructions like- **'Do not lie'** and we give it tools access and an orchestrated planning order.
> *   **VoiceAgent 3:** We add high-end conversational turn detection so it knows when to wait for you to finish a thought instead of cutting you off.


Query this for every agent:

> Plan a 2-day Goa trip with beaches and nightlife, medium budget. Suggest hotels, and price per night.

### How to run a agent

1. Run the setup cells once (install, secrets, catalog, model download, runner).
2. Run **one** agent cell. Leave it running — a spinning cell means the agent is online.
3. Wait until the cell prints `registered worker`.
4. Connect on the website and say the sentence above.
5. **Stop that cell** (⬛) before you run the next agent. The old worker keeps the old code until you stop it.



## Setup — install

This installs the voice stack once. Later agents reuse it. The turn-detection model is only used in VoiceAgent 3, but we install it now so that VoiceAgent 3 does not stop for a download.


In [ ]:
# Install agent packages with flexible modern versions to prevent dependency conflicts
!pip install -q -U livekit-agents livekit-plugins-silero livekit-plugins-simli livekit-plugins-turn-detector livekit-plugins-noise-cancellation python-dotenv pandas openpyxl

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires pandas==2.2.3, but you have pandas 3.1.0 which is incompatible.
google-adk 2.7.1 requires opentelemetry-api<=1.42.1,>=1.39, but you have opentelemetry-api 1.45.1 which is incompatible.
google-adk 2.7.1 requires opentelemetry-sdk<=1.42.1,>=1.39, but you have opentelemetry-sdk 1.45.1 which is incompatible.


## Setup — your LiveKit secrets

Click the **key** icon on the left. Add these secrets and turn **Notebook access** on. Paste the values **without quotes**.

| Secret name | What to paste |
| --- | --- |
| `LIVEKIT_URL` | Project URL, `wss://your-project.livekit.cloud` |
| `LIVEKIT_API_KEY` | API key |
| `LIVEKIT_API_SECRET` | API secret |

Do not paste the token server ID here. That ID goes in the website, not in this notebook.

Speech, the language model, and the voice are billed to **your** LiveKit project. You do not need separate OpenAI, Deepgram, or Cartesia keys.


In [ ]:
import os
from pathlib import Path

try:
    from google.colab import userdata
except ImportError as exc:
    raise SystemExit(
        "Open this notebook in Google Colab. Keys are read from the Colab Secrets panel."
    ) from exc


def secret(name: str) -> str:
    try:
        value = userdata.get(name)
    except Exception as exc:
        raise SystemExit(
            f"Could not read Colab secret {name}. Add it in the key panel and turn Notebook access on. "
            "Do not paste the value into a cell."
        ) from exc
    value = (value or "").strip().strip('"').strip("'")
    if not value:
        raise SystemExit(f"Colab secret {name} is empty.")
    return value


url = secret("LIVEKIT_URL").rstrip("/")
if url.startswith("https://"):
    url = "wss://" + url[len("https://"):]
elif url.startswith("http://"):
    url = "ws://" + url[len("http://"):]
elif "://" not in url:
    url = "wss://" + url
if "sandbox.livekit.io" in url or not url.endswith(".livekit.cloud"):
    raise SystemExit(
        "LIVEKIT_URL must be the project URL, like wss://your-project.livekit.cloud. "
        "Not the token server ID, and not the sandbox.livekit.io URL."
    )

os.environ["LIVEKIT_URL"] = url
os.environ["LIVEKIT_API_KEY"] = secret("LIVEKIT_API_KEY")
os.environ["LIVEKIT_API_SECRET"] = secret("LIVEKIT_API_SECRET")
print("Using LiveKit project:", url.split("://", 1)[-1])
print("API key and secret are loaded. Values are not printed.")


Using LiveKit project: test-agent-5f42hkx0.livekit.cloud
API key and secret are loaded. Values are not printed.


## Setup — the catalog

VoiceAgent 1 do not read this file.

VoiceAgents 2 and 3 may speak a hotel or a city **only if it is in this workbook**. Upload `tripmate_catalog.xlsx` when the next cell asks. It has three sheets we will use:

- `destinations` — city, interests, score, why
- `hotels` — city, name, price per night, rating, budget
- `activities` — city, day, items, interests

If a traveler asks for a city that is not a row, the tool returns nothing. The agent should say so. That is the lesson: the model does not get to fill the gap.

In [ ]:
from pathlib import Path
import pandas as pd

path = Path("/content/tripmate_catalog.xlsx")
if not path.exists():
    path = Path("/content/tripmate_catalog.xlsx")

try:
    from google.colab import files
except ImportError:
    files = None

if not path.exists():
    if files is None:
        raise SystemExit("Put tripmate_catalog.xlsx next to this notebook, then re-run this cell.")
    print("Upload tripmate_catalog.xlsx")
    uploaded = files.upload()
    if not uploaded:
        raise SystemExit("No file uploaded.")
    path = Path("/content/tripmate_catalog.xlsx")
    path.write_bytes(next(iter(uploaded.values())))


def _split_list(value):
    # Excel cells store "beaches, nightlife" as one string. Tools want a list.
    if value is None or (isinstance(value, float) and pd.isna(value)):
        return []
    text = str(value).replace("|", ",").replace(";", ",")
    return [part.strip() for part in text.split(",") if part.strip()]


def _city_key(city: str) -> str:
    return city.strip().lower()


destinations_df = pd.read_excel(path, sheet_name="destinations")
hotels_df = pd.read_excel(path, sheet_name="hotels")
activities_df = pd.read_excel(path, sheet_name="activities")

DESTINATIONS = []
for _, row in destinations_df.iterrows():
    DESTINATIONS.append({
        "city": str(row["city"]).strip(),
        "interests": [item.lower() for item in _split_list(row.get("interests"))],
        "score": float(row["score"]),
        "why": str(row["why"]).strip(),
    })

HOTELS = {}
for _, row in hotels_df.iterrows():
    HOTELS.setdefault(_city_key(str(row["city"])), []).append({
        "name": str(row["name"]).strip(),
        "price_per_night": int(row["price_per_night"]),
        "rating": float(row["rating"]),
        "budget": str(row["budget"]).strip().lower(),
    })

ACTIVITIES = {}
for _, row in activities_df.iterrows():
    ACTIVITIES.setdefault(_city_key(str(row["city"])), []).append({
        "day": int(row["day"]),
        "items": _split_list(row.get("items")),
        "interests": [item.lower() for item in _split_list(row.get("interests"))],
    })

AVAILABLE_CITIES = sorted(place["city"] for place in DESTINATIONS)
print("Catalog:", path.resolve())
print("Cities:", ", ".join(AVAILABLE_CITIES))
print(f"Loaded: {len(DESTINATIONS)} destinations, {sum(len(v) for v in HOTELS.values())} hotels, {sum(len(v) for v in ACTIVITIES.values())} activity plans")


/usr/local/lib/python3.13/dist-packages/pandas/core/arrays/masked.py:63: UserWarning: Pandas requires version '1.5.0' or newer of 'bottleneck' (version '1.4.2' currently installed).
  from pandas.core import (


Catalog: /content/Travel_catalog.xlsx
Cities: Andaman, Goa, Gokarna, Jaipur, Manali, Munnar, Mussoorie, Pondicherry, Rishikesh, Udaipur
Loaded: 10 destinations, 22 hotels, 23 activity plans


## Setup — turn-detection files


VoiceAgent 3 needs a small model file (`languages.json`) so the agent can tell when you have finished speaking. Download it now.
VoiceAgents 1 and 2 do not use it.


In [ ]:
!python -m livekit.agents download-files
print("Model files ready.")


discovered 4 plugin package(s): livekit.plugins.noise_cancellation, livekit.plugins.silero, livekit.plugins.simli, livekit.plugins.turn_detector
downloading files for livekit.plugins.silero
finished downloading files for livekit.plugins.silero
downloading files for livekit.plugins.simli
finished downloading files for livekit.plugins.simli
downloading files for livekit.plugins.turn_detector
NumExpr defaulting to 2 threads.
/usr/local/lib/python3.13/dist-packages/pandas/core/arrays/masked.py:63: UserWarning: Pandas requires version '1.5.0' or newer of 'bottleneck' (version '1.4.2' currently installed).
  from pandas.core import (
HTTP Request: GET https://huggingface.co/api/agent-harnesses "HTTP/1.1 200 OK"
HTTP Request: GET https://huggingface.co/api/models/livekit/turn-detector/revision/v1.2.2-en "HTTP/1.1 200 OK"
HTTP Request: HEAD https://huggingface.co/livekit/turn-detector/resolve/ebcab0c09c2b62d926e92180d364df3aaae68a09/config.json "HTTP/1.1 307 Temporary Redirect"
HTTP Request: H

## Setup — how a VoiceAgent cell stays running

Every VoiceAgent cell ends by calling `run_stage(server)`.

That call does not return. The cell keeps spinning because the worker is online, waiting for someone to join a room. **Spinning is success.** Stopping the cell hangs up the agent.

### What happens when you click Connect

```
Browser clicks Connect
        │
        ▼
LiveKit Cloud creates a Room
        │
        ▼
Cloud checks: "Any registered worker?"
        │  (your spinning cell registered one)
        ▼
Cloud dispatches a Job to your agent
        │
        ▼
Your @server.rtc_session() function runs
        │
        ▼
AgentSession.start() → agent joins the Room
        │
        ▼
Agent is now a participant, same as the user
```

### The voice pipeline inside the agent

```
Mic audio (WebRTC)
   → VAD (Silero, local, ~2 MB)      — speech or silence?
   → STT (Deepgram, cloud API)       — speech → text
   → LLM (GPT-4.1-mini, cloud API)   — text → response
   → TTS (Cartesia, cloud API)       — response → speech
   → Published back via WebRTC        — user hears the agent
```

VAD is the **only model that runs locally**. Everything else is a cloud API call — that is why you need LiveKit keys but not a separate VAD key.

### Interruption

When you talk over the agent: VAD detects your voice → agent stops TTS mid-word → flushes the LLM stream → STT starts on your new speech → pipeline restarts. Under 300 ms.

### Colab detail

`JobExecutorType.THREAD` keeps the agent inside this notebook. LiveKit's default is a separate process, and a function written in a Colab cell cannot be sent to that process.

If you forget to stop the previous VoiceAgent, `run_stage` closes it before starting the new one.


In [ ]:
import logging

from livekit.agents import JobExecutorType

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s %(levelname)s %(message)s",
    force=True,
)

_current_server = None


async def run_stage(new_server, stage_name: str):
    """Start this VoiceAgent and leave the cell running until you press stop."""
    global _current_server
    if _current_server is not None and not _current_server._closed:
        print("Closing the previous VoiceAgent...")
        await _current_server.aclose()
    _current_server = new_server
    print(f"{stage_name} is starting.")
    print("Wait for 'registered worker', then connect on the website and say the Goa sentence.")
    print("Stop this cell before you run the next VoiceAgent.\n")
    await new_server.run(devmode=True)  ## the agent registers itself with LiveKit Cloud —
    # it says 'I am online, send me work' — and then blocks forever, heartbeating every few seconds.


# VoiceAgent 1 — voice chatbot

This is the smallest agent. It can hear, think, and speak.

Look at the instruction. It is one sentence: "You are a helpful voice AI assistant." There is no tool and no spreadsheet. The language model will answer from whatever it already knows, including prices it is happy to invent.

The pipeline:

| Step | Plugin | Runs where | What it does |
| --- | --- | --- | --- |
| VAD | `silero.VAD.load()` | **Locally** (2 MB model in your Colab) | Decides if audio is speech or silence |
| STT | `deepgram/nova-3` | Cloud API | Converts speech → text |
| LLM | `openai/gpt-4.1-mini` | Cloud API | Generates response text |
| TTS | `cartesia/sonic-2` | Cloud API | Converts text → speech |

**Say:** "Plan a 2-day Goa trip with beaches and nightlife, medium budget. What hotel, and what price per night?"

**Listen for:** a fluent answer and a price that is **not** in the Excel sheet. There will be no `TOOL` line in this cell.

That price is the thing we are about to take away.


In [ ]:
import logging

from livekit.agents import (
    Agent,
    AgentServer,
    AgentSession,
    JobContext,
    JobExecutorType,
    RoomInputOptions,
)
from livekit.plugins import noise_cancellation, silero ##

# The entire personality is this string. Nothing else constrains the facts.
class Assistant(Agent):
    def __init__(self) -> None:
        super().__init__(instructions="You are a helpful voice AI assistant.")


server = AgentServer(job_executor_type=JobExecutorType.THREAD)

# When someone clicks Connect on the website, LiveKit Cloud creates a Room.
# Cloud then looks at the registered workers and dispatches a Job to your agent —
# that is the @server.rtc_session() decorator. Your entrypoint function runs,
# creates an AgentSession, and joins the room as a second participant alongside the browser user.
@server.rtc_session()
async def entrypoint(ctx: JobContext):
    # Ears, brain, voice. This pipeline stays the same through VoiceAgent 2.
    session = AgentSession(
        stt="deepgram/nova-3",
        llm="openai/gpt-4.1-mini",
        tts="cartesia/sonic-2",
        vad=silero.VAD.load(),
    )
    await session.start(
        agent=Assistant(),
        room=ctx.room,
        room_input_options=RoomInputOptions(
            noise_cancellation=noise_cancellation.BVC(),
        ),
    )
    await ctx.connect()


await run_stage(server, "VoiceAgent 1 — voice chatbot")


2026-10-29 04:03:50,988 INFO starting worker
2026-10-29 04:03:50,990 INFO plugin registered
2026-10-29 04:03:50,991 INFO preloading plugins
2026-10-29 04:03:51,002 INFO HTTP server listening on :35629


VoiceAgent 1 — voice chatbot is starting.
Wait for 'registered worker', then connect on the website and say the Goa sentence.
Stop this cell before you run the next VoiceAgent.



2026-10-29 04:03:51,247 INFO registered worker
2026-10-29 04:04:38,172 INFO received job request
2026-10-29 04:04:38,196 WARNING no warmed process available for job, waiting for one to be created
2026-10-29 04:04:38,199 INFO initializing job runner
2026-10-29 04:04:38,748 INFO job runner initialized
2026-10-29 04:04:38,928 WARNING RoomInputOptions and RoomOutputOptions are deprecated, use RoomOptions instead
2026-10-29 04:04:38,959 INFO adaptive interruption is disabled by default in production mode
2026-10-29 04:04:38,972 WARNING event loop blocked for 203ms at "/usr/local/lib/python3.13/dist-packages/onnxruntime/capi/onnxruntime_inference_collection.py", line 789, in _create_inference_session; synchronous work on the agent loop delays audio and turn handling, move it to a thread or an async client
2026-10-29 04:04:39,089 INFO livekit_signaling::signal_stream:53:livekit_signaling::signal_stream - connecting to wss://test-agent-5f42hkx0.otoronto1a.production.livekit.cloud/rtc/v1
2026-1

CancelledError: 

#VoiceAgent 2 — three tools and orchestrated planning

Stop VoiceAgent 1 first.

**Change:** We introduce three Python tools: `search_destinations`, `find_hotels`, and `find_activities`.

The tools:
- `find_hotels` — find hotels in a city from the hotels spreadsheet.
- `search_destinations` — ranks cities from the destinations sheet by traveler interests
- `find_activities` — builds a day-by-day plan from the activities sheet.


We also update our instruction prompt to act as an orchestrator, guiding the model on the planning sequence: searching destinations first, then querying hotels, and finally getting the daily activities.

Three things makes these functions a tool such that model can call:

1. The decorator `@function_tool`.
2. The docstring. That text is what the model reads to decide when to call it.
3. The `Args` lines. Those become the arguments the model must fill in.

**Say the same Goa sentence:**
"Plan a 2-day Goa trip with beaches and nightlife, medium budget. What hotel, and what price per night?"

**Listen for:** The agent replying with actual facts and prices from our uploaded catalog sheet instead of hallucinating details. Observe the tool lines appearing in the execution console in proper sequence.

This is a tool-using agent with a plan. Make sure to observe, if you pause in the middle of the sentence, it may still jump in. We will fix this in next step.

In [ ]:
from livekit.agents import (
    Agent,
    AgentServer,
    AgentSession,
    JobContext,
    JobExecutorType,
    RoomInputOptions,
    RunContext,
    function_tool,
)
from livekit.plugins import noise_cancellation, silero


def _normalize_interests(interests: list[str]) -> list[str]:
    cleaned = []
    for interest in interests:
        value = interest.strip().lower()
        if value in {"beach", "beachside"}:
            value = "beaches"
        if value in {"night life", "party", "parties", "clubs"}:
            value = "nightlife"
        if value in {"hill", "hills", "hill station"}:
            value = "mountains"
        if value:
            cleaned.append(value)
    return cleaned


class TripMate(Agent):
    def __init__(self) -> None:
        super().__init__(
            instructions=(
                "You are TripMate, a voice travel planner for trips in India. "
                "Speak in short spoken sentences. "
                "Never invent a city, hotel, price, or activity. "
                "Call search_destinations first when the traveler describes a trip. "
                "After you name a city, call find_hotels for that city, then call find_activities. "
                "For a medium budget, prefer the lower-priced well-rated hotel. "
                "If a tool says the city is not in the catalog, say that and name a city that is."
            ),
        )

    @function_tool
    async def search_destinations(
        self,
        context: RunContext,
        interests: list[str],
        duration: int,
        budget: str = "medium",
    ) -> dict:
        """Rank cities from the workshop spreadsheet for this trip.

        Args:
            interests: What the traveler wants, such as beaches or nightlife.
            duration: Trip length in days.
            budget: low, medium, or high.
        """
        print(f"TOOL search_destinations interests={interests} duration={duration} budget={budget}")
        wanted = set(_normalize_interests(interests))
        ranked = []
        for place in DESTINATIONS:
            overlap = wanted & set(place["interests"])
            if wanted and not overlap:
                continue
            ranked.append({
                "city": place["city"],
                "score": place["score"],
                "matched_interests": sorted(overlap),
                "why": place["why"],
                "duration_days": duration,
                "budget": budget,
            })
        ranked.sort(key=lambda item: item["score"], reverse=True)
        return {"destinations": ranked[:3], "available_cities": AVAILABLE_CITIES}

    @function_tool
    async def find_hotels(
        self,
        context: RunContext,
        city: str,
        nights: int,
        budget: str = "medium",
    ) -> dict:
        """Find hotels in a city from the workshop spreadsheet.

        Args:
            city: Destination city, such as Goa.
            nights: Number of nights to stay.
            budget: low, medium, or high.
        """
        print(f"TOOL find_hotels city={city} nights={nights} budget={budget}")
        stays = HOTELS.get(_city_key(city), [])
        if not stays:
            return {
                "city": city,
                "hotels": [],
                "note": f"No hotels in the catalog for {city}.",
                "available_cities": AVAILABLE_CITIES,
            }
        allowed = {budget.lower()}
        if budget.lower() == "medium":
            allowed.add("high")
        chosen = [hotel for hotel in stays if hotel["budget"] in allowed] or stays
        return {
            "city": city,
            "nights": nights,
            "hotels": [
                {
                    "name": hotel["name"],
                    "price_per_night_inr": hotel["price_per_night"],
                    "stay_total_inr": hotel["price_per_night"] * nights,
                    "rating": hotel["rating"],
                }
                for hotel in chosen
            ],
        }

    @function_tool
    async def find_activities(
        self,
        context: RunContext,
        city: str,
        interests: list[str],
        duration: int,
    ) -> dict:
        """Build a day-by-day activity plan from the workshop spreadsheet.

        Args:
            city: Destination city, such as Goa or Mussoorie.
            interests: Interests to prefer, such as beaches and nightlife.
            duration: Number of days to plan.
        """
        print(f"TOOL find_activities city={city} interests={interests} duration={duration}")
        catalog = ACTIVITIES.get(_city_key(city), [])
        if not catalog:
            return {
                "city": city,
                "days": [],
                "note": f"No activities in the catalog for {city}.",
                "available_cities": AVAILABLE_CITIES,
            }
        wanted = set(_normalize_interests(interests))
        days = []
        for plan in catalog:
            if plan["day"] > duration:
                continue
            if wanted and not (wanted & set(plan["interests"])):
                continue
            days.append({"day": plan["day"], "items": plan["items"]})
        if not days:
            days = [
                {"day": plan["day"], "items": plan["items"]}
                for plan in catalog
                if plan["day"] <= duration
            ]
        return {"city": city, "duration_days": duration, "days": days}


server = AgentServer(job_executor_type=JobExecutorType.THREAD)


@server.rtc_session()
async def entrypoint(ctx: JobContext):
    session = AgentSession(
        stt="deepgram/nova-3",
        llm="openai/gpt-4.1-mini",
        tts="cartesia/sonic-2:9626c31c-bec5-4cca-baa8-f8ba9e84c8bc",
        vad=silero.VAD.load(),
        turn_detection="vad",
    )
    await session.start(
        agent=TripMate(),
        room=ctx.room,
        room_input_options=RoomInputOptions(
            noise_cancellation=noise_cancellation.BVC(),
        ),
    )
    await ctx.connect()


await run_stage(server, "VoiceAgent 2 — three tools and planning")

2026-10-29 04:05:15,471 INFO shutting down worker


Closing the previous VoiceAgent...


2026-10-29 04:05:16,051 WARNING livekit_ffi::server:279:livekit_ffi::server - Attempted to drop unknown FFI handle: 22
2026-10-29 04:05:16,091 INFO livekit::room:1813:livekit::room - Disconnected from room "sbx-1vvqd1-7Smw9D947wYLu3rTojSF7m" with reason: ClientInitiated
2026-10-29 04:05:16,226 INFO starting worker
2026-10-29 04:05:16,227 INFO plugin registered
2026-10-29 04:05:16,228 INFO preloading plugins
2026-10-29 04:05:16,230 INFO HTTP server listening on :42317
2026-10-29 04:05:16,355 INFO registered worker


VoiceAgent 2 — three tools and planning is starting.
Wait for 'registered worker', then connect on the website and say the Goa sentence.
Stop this cell before you run the next VoiceAgent.



2026-10-29 04:05:31,274 INFO received job request
2026-10-29 04:05:31,298 WARNING no warmed process available for job, waiting for one to be created
2026-10-29 04:05:31,301 INFO initializing job runner
2026-10-29 04:05:31,305 INFO job runner initialized
2026-10-29 04:05:31,367 WARNING turn_detection is deprecated and will be removed in v2.0. Use turn_handling=TurnHandlingOptions(...) instead
2026-10-29 04:05:31,405 WARNING RoomInputOptions and RoomOutputOptions are deprecated, use RoomOptions instead
2026-10-29 04:05:31,414 INFO adaptive interruption is disabled by default in production mode
2026-10-29 04:05:31,455 INFO livekit_signaling::signal_stream:53:livekit_signaling::signal_stream - connecting to wss://test-agent-5f42hkx0.otoronto1a.production.livekit.cloud/rtc/v1
2026-10-29 04:05:31,488 INFO HTTP Request: GET https://agent-gateway.livekit.cloud/v1/models "HTTP/1.1 200 OK"
2026-10-29 04:05:32,003 INFO RoomIO linked to participant
2026-10-29 04:05:34,715 INFO HTTP Request: POST h

TOOL search_destinations interests=['nightlife', 'beaches'] duration=2 budget=medium


2026-10-29 04:05:40,755 INFO HTTP Request: POST https://agent-gateway.livekit.cloud/v1/chat/completions "HTTP/1.1 200 OK"


TOOL find_hotels city=Goa nights=2 budget=medium
TOOL find_activities city=Goa interests=['nightlife', 'beaches'] duration=2


2026-10-29 04:05:41,487 INFO HTTP Request: POST https://agent-gateway.livekit.cloud/v1/chat/completions "HTTP/1.1 200 OK"
2026-10-29 04:05:51,203 INFO closing agent session due to participant disconnect (disable via `RoomInputOptions.close_on_disconnect=False`)


CancelledError: 

#VoiceAgent 3 — turn detection & conversational intelligence

Stop VoiceAgent 2 first.

**Change:** We configure the advanced model-driven conversational turn-detector (`inference.TurnDetector()`).

Until now, the agent used voice-activity detection (VAD), which sometimes cut you off on short pauses. With the new turn detector, it looks at your linguistic state and semantic flow before replying.

**Say the same Goa sentence, but pause** after "Plan a 2-day Goa trip..." for about two seconds, then finish with "with beaches and nightlife, medium budget."

**Listen for:** The agent should wait patiently during your pause and only initiate the planning tools once you complete the entire command!


That is the whole arc:

- VoiceAgent 1 was a voice chatbot.
- VoiceAgent 2 was a tool-using agent and planning orchestrator.
- VoiceAgent 3 is a conversational agent: same tools, plus a sense of when you are finished talking.

In [ ]:
import os

from livekit.agents import (
    Agent,
    AgentServer,
    AgentSession,
    JobContext,
    JobExecutorType,
    RoomInputOptions,
    RunContext,
    TurnHandlingOptions,
    function_tool,
    inference,
)
from livekit.plugins import noise_cancellation, silero
from livekit.plugins.turn_detector.multilingual import MultilingualModel

# We import the simli plugin at the top level of the cell so it is registered on the main thread
try:
    from livekit.plugins import simli as simli_plugin
except ImportError:
    simli_plugin = None

# Helper function to map common user synonyms to standard catalog interests
def _normalize_interests(interests: list[str]) -> list[str]:
    cleaned = []
    for interest in interests:
        value = interest.strip().lower()
        if value in {"beach", "beachside"}:
            value = "beaches"
        if value in {"night life", "party", "parties", "clubs"}:
            value = "nightlife"
        if value in {"hill", "hills", "hill station"}:
            value = "mountains"
        if value:
            cleaned.append(value)
    return cleaned

# Define the TripMate agent with customized system prompt instructions
class TripMate(Agent):
    def __init__(self) -> None:
        super().__init__(
            instructions=(
                "You are TripMate, a voice travel planner for trips in India. "
                "Speak in short spoken sentences. "
                "Never invent a city, hotel, price, or activity. "
                "Call search_destinations first when the traveler describes a trip. "
                "After you name a city, call find_hotels for that city, then call find_activities. "
                "For a medium budget, prefer the lower-priced well-rated hotel. "
                "If a tool says the city is not in the catalog, say that and name a city that is."
            ),
        )

    # Tool 1: Matches traveler interests and scores the best destination cities
    @function_tool
    async def search_destinations(
        self,
        context: RunContext,
        interests: list[str],
        duration: int,
        budget: str = "medium",
    ) -> dict:
        """Rank cities from the workshop spreadsheet for this trip.

        Args:
            interests: What the traveler wants, such as beaches or nightlife.
            duration: Trip length in days.
            budget: low, medium, or high.
        """
        print(f"TOOL search_destinations interests={interests} duration={duration} budget={budget}")
        wanted = set(_normalize_interests(interests))
        ranked = []
        for place in DESTINATIONS:
            overlap = wanted & set(place["interests"])
            if wanted and not overlap:
                continue
            ranked.append({
                "city": place["city"],
                "score": place["score"],
                "matched_interests": sorted(overlap),
                "why": place["why"],
                "duration_days": duration,
                "budget": budget,
            })
        ranked.sort(key=lambda item: item["score"], reverse=True)
        return {"destinations": ranked[:3], "available_cities": AVAILABLE_CITIES}

    # Tool 2: Fetches matching hotel properties based on budget and location from the spreadsheet
    @function_tool
    async def find_hotels(
        self,
        context: RunContext,
        city: str,
        nights: int,
        budget: str = "medium",
    ) -> dict:
        """Find hotels in a city from the workshop spreadsheet.

        Args:
            city: Destination city, such as Goa.
            nights: Number of nights to stay.
            budget: low, medium, or high.
        """
        print(f"TOOL find_hotels city={city} nights={nights} budget={budget}")
        stays = HOTELS.get(_city_key(city), [])
        if not stays:
            return {
                "city": city,
                "hotels": [],
                "note": f"No hotels in the catalog for {city}.",
                "available_cities": AVAILABLE_CITIES,
            }
        allowed = {budget.lower()}
        if budget.lower() == "medium":
            allowed.add("high")
        chosen = [hotel for hotel in stays if hotel["budget"] in allowed] or stays
        return {
            "city": city,
            "nights": nights,
            "hotels": [
                {
                    "name": hotel["name"],
                    "price_per_night_inr": hotel["price_per_night"],
                    "stay_total_inr": hotel["price_per_night"] * nights,
                    "rating": hotel["rating"],
                }
                for hotel in chosen
            ],
        }

    # Tool 3: Extracts Day-by-Day activities from the spreadsheet catalog
    @function_tool
    async def find_activities(
        self,
        context: RunContext,
        city: str,
        interests: list[str],
        duration: int,
    ) -> dict:
        """Build a day-by-day activity plan from the workshop spreadsheet.

        Args:
            city: Destination city, such as Goa or Mussoorie.
            interests: Interests to prefer, such as beaches and nightlife.
            duration: Number of days to plan.
        """
        print(f"TOOL find_activities city={city} interests={interests} duration={duration}")
        catalog = ACTIVITIES.get(_city_key(city), [])
        if not catalog:
            return {
                "city": city,
                "days": [],
                "note": f"No activities in the catalog for {city}.",
                "available_cities": AVAILABLE_CITIES,
            }
        wanted = set(_normalize_interests(interests))
        days = []
        for plan in catalog:
            if plan["day"] > duration:
                continue
            if wanted and not (wanted & set(plan["interests"])):
                continue
            days.append({"day": plan["day"], "items": plan["items"]})
        if not days:
            days = [
                {"day": plan["day"], "items": plan["items"]}
                for plan in catalog
                if plan["day"] <= duration
            ]
        return {"city": city, "duration_days": duration, "days": days}


# Initialize the AgentServer in thread mode to run within Colab notebook environment
server = AgentServer(job_executor_type=JobExecutorType.THREAD)


# This entrypoint is triggered automatically whenever a user connects to a LiveKit room
@server.rtc_session()
async def entrypoint(ctx: JobContext):
    # Initialize AgentSession with advanced conversational turn-detector
    session = AgentSession(
        stt="deepgram/nova-3",
        llm="openai/gpt-4.1-mini",
        tts="cartesia/sonic-2:9626c31c-bec5-4cca-baa8-f8ba9e84c8bc",
        vad=silero.VAD.load(),
        # Advanced turn handling configured here to wait out semantic pauses natively
        turn_handling=TurnHandlingOptions(
            turn_detection=inference.TurnDetector(),
            mode="fixed",
            min_delay=4,
            max_delay=6.0,
        ),
    )

    agent = TripMate()

    # If the bonus Simli cell was run, attach the generated dynamic face/avatar
    simli_key = os.environ.get("SIMLI_API_KEY", "").strip()
    if simli_key and simli_plugin:
        face_id = os.environ.get("SIMLI_FACE_ID", "cace3ef7-a4c4-425d-a8cf-a5358eb0c427")
        avatar = simli_plugin.AvatarSession(
            simli_config=simli_plugin.SimliConfig(api_key=simli_key, face_id=face_id)
        )
        print(f"Simli avatar ON — face {face_id}")
        await avatar.start(session, room=ctx.room)
        await avatar.wait_for_join()

    # Start the session and join the LiveKit room
    await session.start(
        agent=agent,
        room=ctx.room,
        room_input_options=RoomInputOptions(
            noise_cancellation=noise_cancellation.BVC(),
        ),
    )
    await ctx.connect()


# Register with the Livekit Cloud to bring the VoiceAgent 3 worker online
await run_stage(server, "VoiceAgent 3 — turn detection conversational agent")

/tmp/ipykernel_1020/2569184031.py:16: DeprecationWarning: `livekit.plugins.turn_detector` is deprecated and will be removed in a future release. Use `livekit.agents.inference.TurnDetector` instead.
  from livekit.plugins.turn_detector.multilingual import MultilingualModel
2026-10-29 04:06:04,390 INFO shutting down worker


Closing the previous VoiceAgent...


2026-10-29 04:06:04,898 WARNING livekit_ffi::server:279:livekit_ffi::server - Attempted to drop unknown FFI handle: 2393
2026-10-29 04:06:04,987 INFO starting worker
2026-10-29 04:06:04,988 INFO plugin registered
2026-10-29 04:06:04,989 INFO plugin registered
2026-10-29 04:06:04,990 INFO plugin registered
2026-10-29 04:06:04,991 INFO plugin registered
2026-10-29 04:06:04,992 INFO preloading plugins
2026-10-29 04:06:04,993 INFO starting inference executor


VoiceAgent 3 — turn detection conversational agent is starting.
Wait for 'registered worker', then connect on the website and say the Goa sentence.
Stop this cell before you run the next VoiceAgent.



2026-10-29 04:06:08,244 INFO initializing process
2026-10-29 04:06:29,687 INFO process initialized
2026-10-29 04:06:29,695 INFO HTTP server listening on :46511
2026-10-29 04:06:29,813 INFO registered worker
2026-10-29 04:06:30,071 INFO received job request
2026-10-29 04:06:30,095 WARNING no warmed process available for job, waiting for one to be created
2026-10-29 04:06:30,098 INFO initializing job runner
2026-10-29 04:06:30,101 INFO job runner initialized
2026-10-29 04:06:30,249 WARNING RoomInputOptions and RoomOutputOptions are deprecated, use RoomOptions instead
2026-10-29 04:06:30,259 INFO adaptive interruption is disabled by default in production mode
2026-10-29 04:06:30,265 WARNING event loop blocked for 146ms at "/usr/local/lib/python3.13/dist-packages/onnxruntime/capi/onnxruntime_inference_collection.py", line 789, in _create_inference_session; synchronous work on the agent loop delays audio and turn handling, move it to a thread or an async client
2026-10-29 04:06:30,302 INFO 

TOOL search_destinations interests=['nightlife', 'beaches'] duration=2 budget=medium


2026-10-29 04:06:49,356 INFO HTTP Request: POST https://agent-gateway.livekit.cloud/v1/chat/completions "HTTP/1.1 200 OK"


TOOL find_hotels city=Goa nights=2 budget=medium
TOOL find_activities city=Goa interests=['nightlife', 'beaches'] duration=2


2026-10-29 04:06:50,206 INFO HTTP Request: POST https://agent-gateway.livekit.cloud/v1/chat/completions "HTTP/1.1 200 OK"
2026-10-29 04:06:56,062 INFO closing agent session due to participant disconnect (disable via `RoomInputOptions.close_on_disconnect=False`)


CancelledError: 

# Bonus — Simli avatar (optional)

This cell is **optional**. If you run this cell, VoiceAgent 3 will show a Simli avatar on the website. If you skip it, nothing breaks — you just see the plane icon.

### Setup

1. Add a Colab Secret called `SIMLI_API_KEY` with your Simli API key (22 characters). Turn **Notebook access** on.
2. Optionally add `SIMLI_FACE_ID` if you want a different face. The default is a young woman.
3. Run this cell, then **re-run VoiceAgent 3**. The avatar appears on the website.

The API key is **not** the face ID. The key is shorter (22 characters). The face ID is a UUID (36 characters).

You can register for Simli and get your API key from their [website](https://www.simli.com/)

In [ ]:
import os

try:
    from google.colab import userdata
    simli_key = userdata.get("SIMLI_API_KEY")
except Exception:
    simli_key = None

if simli_key:
    simli_key = simli_key.strip().strip('"').strip("'")
    os.environ["SIMLI_API_KEY"] = simli_key

    # Optional face ID override
    try:
        face_id = userdata.get("SIMLI_FACE_ID")
        if face_id:
            os.environ["SIMLI_FACE_ID"] = face_id.strip().strip('"').strip("'")
    except Exception:
        pass

    face = os.environ.get("SIMLI_FACE_ID", "cace3ef7-a4c4-425d-a8cf-a5358eb0c427")
    print(f"✅ Simli avatar enabled — face {face}")
    print(f"   SIMLI_API_KEY length: {len(simli_key)}")
    if len(simli_key) > 30:
        print("   ⚠️  That key looks too long. The API key is ~22 characters. The face ID is 36. Make sure you pasted the API key, not the face ID.")
    print("\n Now re-run VoiceAgent 3. The avatar will appear on the website.")
else:
    print("ℹ️  No SIMLI_API_KEY found in Colab Secrets. Skipping avatar.")
    print("   All VoiceAgents still work — you just won't see a face.")


✅ Simli avatar enabled — face cace3ef7-a4c4-425d-a8cf-a5358eb0c427
   SIMLI_API_KEY length: 22

 Now re-run Stage 5. The avatar will appear on the website.


## What you just walked


Same website. Same Goa sentence. Three agents.

1. **VoiceAgent 1 - Voice chatbot** — it could talk, and it invented the price.
2. **VoiceAgent 2 - Tools and orchestrator** — `search_destinations`, then `find_hotels`, then `find_activities`. The model followed an order and planned the whole trip from the sheet.
3. **VoiceAgent 3 - A conversation** — the turn detector waited out a pause. The tools did not change.


---
